# Workshop 1 - Gold KPI Package (solution)

![Workshop success criteria](../../assets/images/workshop_success_criteria.png)

## Scenario

RetailHub needs a new daily channel reporting slice for Power BI. This solution shows one defensible implementation of the Gold table, KPI definitions, data-quality checks, reconciliation and decision-log entry.

## Objectives

After reviewing this solution you will be able to:

- explain why the daily channel object is materialized as a Gold table,
- calculate Average Order Value, Margin Rate % and Completed Share correctly,
- validate data-quality issues with examples,
- reconcile daily-channel and monthly aggregates,
- document the model decision in a reusable way.

## Prerequisites

Before running, complete:

1. `setup/00_setup.ipynb`
2. `notebooks/demo/day1_02_lakehouse_delta_gold.ipynb`
3. `notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb`

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this workshop.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before starting the tasks.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream demo notebook has not created the required Gold objects.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_02_lakehouse_delta_gold.ipynb (for fact_sales_dashboard) and notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly)")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_02_lakehouse_delta_gold.ipynb (for fact_sales_dashboard) and notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly)\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## Success Criteria

You are done when:

1. `gold.fact_sales_dashboard_channel_daily` exists with the documented grain.
2. Average Order Value, Margin Rate % and Completed Share by Region are
   calculated from Gold - these are NOT the same KPI the Day 1 demo already showed
   you (Revenue, Gross Margin, Return Rate, Orders) - you are extending the
   dictionary, not repeating it.
3. At least three data-quality issues are named with example rows, using the
   bad-data-lab patterns from the data generator.
4. Two aggregates that should agree are reconciled, and any mismatch is
   explained.
5. The decision log (`docs/templates/decision-log.md`) has a real, filled-in
   row for this workshop's reporting-source decision.
6. The self-check cell passes.

## Prerequisite Chain

This workshop is the last link in a chain. If the pre-check above failed,
walk back through the chain in order:

1. `setup/00_pre_config.ipynb`
2. `data/generate_training_dataset.ipynb`
3. `notebooks/demo/day1_02_lakehouse_delta_gold.ipynb` - build the first Gold detail contract.
4. `notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb` - uruchom Gold modeling demo najpierw,
   jesli `gold.fact_sales_dashboard` lub `gold.fact_sales_dashboard_monthly`
   nie istnieja. Ten warsztat startuje dokladnie tam, gdzie demo Gold semantic layer sie konczy.

## The 5 Tasks

1. **Zadanie 1 - zbuduj reporting table.** Build a new Gold object,
   `gold.fact_sales_dashboard_channel_daily`, one level more granular in time
   (daily, not monthly) and narrower in dimensions (channel only) than
   anything the Day 1 demo built - a realistic "BI asked for a daily channel view"
   request.
2. **Zadanie 2 - zdefiniuj KPI.** Define four KPI that the Day 1 demo dictionary
   does NOT already contain: Average Order Value, Margin Rate %, Completed
   Share by Region and Revenue per Channel-Day.
3. **Zadanie 3 - znajdz data quality issues.** Find at least three
   data-quality issues directly in `gold.fact_sales_dashboard`, with example
   rows, using the bad-data-lab patterns.
4. **Zadanie 4 - zrob reconciliation.** Compare your new daily-channel
   aggregate against `gold.fact_sales_dashboard_monthly` for the same scope
   and prove they agree (or explain why they do not).
5. **Zadanie 5 - wypelnij decision log.** Fill a real row in the decision log
   for the table-vs-view-vs-aggregate choice you made in Task 1.

## Zadanie 1 - zbuduj reporting table

**Dlaczego:** The Day 1 Gold modeling demo already gave us a monthly aggregate
(`fact_sales_dashboard_monthly`) and a segment/category aggregate (the Bonus
`fact_sales_dashboard_segment_monthly`). Neither answers "how did each
channel perform yesterday?" - a daily, channel-only grain is a genuinely new
reporting need, not a copy of what already exists. We build it as a table
(not a view) because BI will hit it repeatedly and the source detail table
is large enough that repeated re-aggregation on every Power BI query would
be wasteful.

**Alternative considered:** a view would avoid a refresh job, but every
DirectQuery page render would re-scan and re-aggregate
`fact_sales_dashboard`. For a daily-refreshed channel dashboard, the
materialized-table cost (one scheduled job) is cheaper than the
query-time cost (many ad-hoc scans), so we choose table + scheduled refresh.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD}.fact_sales_dashboard_channel_daily
COMMENT 'Workshop 1 reporting table: one row per order_date x channel, materialized as a table (not a view) because it is refreshed on a schedule and queried repeatedly by Power BI - see decision log.'
AS
SELECT
  order_date,
  channel,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) AS gross_margin
FROM {GOLD}.fact_sales_dashboard
GROUP BY order_date, channel
""")
print("[OK] gold.fact_sales_dashboard_channel_daily created")

spark.sql(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT (order_date, channel)) AS distinct_grain_keys
FROM {GOLD}.fact_sales_dashboard_channel_daily
""").show()

## Zadanie 2 - zdefiniuj KPI

**Dlaczego:** Average Order Value needs revenue divided by a DISTINCT order
count (the same distinct-count trap the Day 1 demo flagged) - not a per-line
average. Margin Rate % is a ratio, not a sum, so it must be computed after
aggregation, never pre-averaged per row. Completed Share by Region needs the
denominator to include both completed and non-completed orders for that
region, otherwise the "share" is meaningless (always 100%).

In [ ]:
spark.sql(f"""
SELECT
  ROUND(
    SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END)
    / NULLIF(COUNT(DISTINCT CASE WHEN is_completed THEN order_id END), 0),
    2
  ) AS avg_order_value,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct
FROM {GOLD}.fact_sales_dashboard
""").show()

In [ ]:
spark.sql(f"""
SELECT
  customer_region,
  COUNT(DISTINCT order_id) AS all_orders,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(
    100.0 * COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)
    / NULLIF(COUNT(DISTINCT order_id), 0),
    2
  ) AS completed_share_pct
FROM {GOLD}.fact_sales_dashboard
GROUP BY customer_region
ORDER BY completed_share_pct DESC
""").show()

In [ ]:
spark.sql(f"""
SELECT order_date, channel, revenue,
       ROUND(revenue / NULLIF(completed_orders, 0), 2) AS revenue_per_completed_order
FROM {GOLD}.fact_sales_dashboard_channel_daily
ORDER BY order_date DESC, channel
LIMIT 10
""").show()

## Zadanie 3 - znajdz data quality issues

**Dlaczego:** The demo quality gate checks the BI-ready Gold object. Here we check
the same families of issues directly against `gold.fact_sales_dashboard` to show
participants that a left join does not "clean" orphan references - it turns
them into NULL dimension attributes that silently break any GROUP BY on
`segment`, `customer_region`, `category` or `subcategory`. That is a
different, Gold-specific symptom of the same Silver-layer root cause.

In [ ]:
spark.sql(f"""
SELECT 'missing_unit_price' AS issue_type, COUNT(*) AS issue_count
FROM {GOLD}.fact_sales_dashboard WHERE unit_price IS NULL
UNION ALL
SELECT 'invalid_status', COUNT(*)
FROM {GOLD}.fact_sales_dashboard WHERE status NOT IN ('Completed','Cancelled','Returned')
UNION ALL
SELECT 'future_order_date', COUNT(*)
FROM {GOLD}.fact_sales_dashboard WHERE order_date > current_date()
UNION ALL
SELECT 'null_customer_region_after_join', COUNT(*)
FROM {GOLD}.fact_sales_dashboard WHERE customer_region IS NULL
UNION ALL
SELECT 'null_category_after_join', COUNT(*)
FROM {GOLD}.fact_sales_dashboard WHERE category IS NULL
ORDER BY issue_count DESC
""").show()

In [ ]:
spark.sql(f"""
SELECT line_id, order_id, order_date, status, customer_region, category, unit_price, line_revenue
FROM {GOLD}.fact_sales_dashboard
WHERE unit_price IS NULL
   OR status NOT IN ('Completed','Cancelled','Returned')
   OR order_date > current_date()
   OR customer_region IS NULL
   OR category IS NULL
ORDER BY order_date DESC
LIMIT 20
""").show(truncate=False)

**Alternative considered:** these checks could instead run against
`silver.order_lines`/`silver.customers`/`silver.products` directly (as
the data generator does) to catch issues before they reach Gold. We
deliberately check Gold here too, because a Gold-level check catches a
different failure mode: a join bug or a stale dimension table that
introduces NULLs even when Silver itself is clean. Production setups should
have both - Silver gate AND Gold spot-check.

## Zadanie 4 - zrob reconciliation

**Dlaczego:** `fact_sales_dashboard_channel_daily` (Task 1) and
`fact_sales_dashboard_monthly` (Day 1 demo) are two independently aggregated
views of the same detail table, sliced on different dimensions (channel+day
vs region+category+channel+month). If we roll the daily-channel table up to
month+channel and compare it against the monthly table rolled down to
month+channel, the revenue totals must match exactly - this proves both
aggregates trace back to the same detail rows with no double-counting or
silent filter drift.

In [ ]:
spark.sql(f"""
WITH from_daily AS (
  SELECT date_format(order_date, 'yyyy-MM') AS year_month, channel,
         ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_channel_daily
  GROUP BY date_format(order_date, 'yyyy-MM'), channel
),
from_monthly AS (
  SELECT year_month, channel, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
  GROUP BY year_month, channel
)
SELECT
  d.year_month, d.channel,
  d.revenue AS daily_rollup_revenue,
  m.revenue AS monthly_table_revenue,
  ROUND(d.revenue - m.revenue, 2) AS diff
FROM from_daily d
JOIN from_monthly m ON d.year_month = m.year_month AND d.channel = m.channel
ORDER BY ABS(d.revenue - m.revenue) DESC
LIMIT 15
""").show()

**Alternative considered:** we could reconcile against `gold.fact_sales`
(the generator's raw fact) instead of `fact_sales_dashboard_monthly`. That
would also work, but it would only prove the join in `fact_sales_dashboard`
is correct - it would not prove the two Gold-layer aggregates participants
actually hand to BI agree with each other, which is the failure mode BI
developers hit in practice (two dashboard tiles built from different
aggregates showing different numbers).

## Zadanie 5 - wypelnij decision log

**Dlaczego:** a decision without a written reason is not reusable - the next
person rebuilds the table-vs-view-vs-aggregate debate from scratch. This row
is the actual `docs/templates/decision-log.md` entry for the Task 1 choice,
filled as a worked example.

| Date | Decision | Options considered | Chosen option | Reason | Consequence |
|---|---|---|---|---|---|
| 2026-06-23 | Reporting source for daily channel view | view / table / materialized aggregate | table (`fact_sales_dashboard_channel_daily`), scheduled refresh | repeated Power BI queries against a daily grain would re-scan and re-aggregate `fact_sales_dashboard` on every page load; a table makes the cost predictable and the refresh schedule explicit | requires a scheduled job (e.g. Lakeflow/Workflow) to keep the table current; consumers see data as of the last refresh, not real-time |

## Self-check

In [ ]:
assert spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard_channel_daily"), "Missing Task 1 reporting table"

grain = spark.sql(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT (order_date, channel)) AS distinct_keys
FROM {GOLD}.fact_sales_dashboard_channel_daily
""").first()
assert grain["rows"] == grain["distinct_keys"], "Channel-daily table grain is not unique on (order_date, channel)"

recon = spark.sql(f"""
WITH from_daily AS (
  SELECT date_format(order_date, 'yyyy-MM') AS year_month, channel, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_channel_daily
  GROUP BY date_format(order_date, 'yyyy-MM'), channel
),
from_monthly AS (
  SELECT year_month, channel, ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
  GROUP BY year_month, channel
)
SELECT MAX(ABS(d.revenue - m.revenue)) AS max_diff
FROM from_daily d JOIN from_monthly m ON d.year_month = m.year_month AND d.channel = m.channel
""").first()["max_diff"]
assert recon < 0.01, f"Task 4 reconciliation does not hold: max diff {recon}"

print("[OK] Workshop 1 self-check passed")

## Summary

This solution shows a complete Gold KPI package: a materialized daily-channel table, KPI definitions with correct distinct-count logic, data-quality evidence, reconciliation and a decision-log entry.

Use it as the trainer reference for discussing why Gold design decisions belong upstream from individual Power BI reports.